# 02 - Complete Dataset Audit
## E-commerce Sales Analysis & Business Intelligence Dashboard
**Dataset:** UCI Online Retail II (`Year 2009-2010` & `Year 2010-2011`)  
**Tools:** Python, Pandas, NumPy  

---

### Audit Objectives & Principles
Before performing any data cleaning, filtering, or feature engineering, we must perform a complete data-quality audit across the entire two-year historical dataset (December 2009 – December 2011).

1. **Full Dataset Ingestion:** Ingest both Excel sheets (`Year 2009-2010` and `Year 2010-2011`) and concatenate them into a single comprehensive DataFrame named `df_raw`.
2. **Data Preservation:** The original raw dataset is strictly preserved without modifying, deleting, or imputing any records.
3. **Objective Anomaly Profiling:** Quantify missing values, duplicates, cancellations, negative/zero quantities, and negative/zero unit prices.
4. **Context-Aware Business Analysis:** Distinguish between genuine customer transactions (e.g. returns/cancellations) and administrative entries (e.g. inventory adjustments, bad debt adjustments, samples).
5. **Quality Summary:** Produce an executive data-quality summary table with exact metrics to guide downstream cleaning decisions.

In [1]:
import os
import numpy as np
import pandas as pd

# Display settings for professional auditing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("Environment initialized: Pandas and NumPy loaded successfully.")

Environment initialized: Pandas and NumPy loaded successfully.


## 1. Ingestion: Loading Both Sheets into `df_raw`

The dataset `online_retail_II.xlsx` contains two distinct sheets covering two calendar periods:
- **Sheet 1:** `Year 2009-2010` (December 1, 2009 – December 9, 2010)
- **Sheet 2:** `Year 2010-2011` (December 1, 2010 – December 9, 2011)

We load both sheets and concatenate them into a single consolidated DataFrame named `df_raw` using `pd.concat(..., ignore_index=True)`. We preserve all original records intact.

In [2]:
# Define dataset path
file_path = '../data/raw/online_retail_II.xlsx' if os.path.exists('../data/raw/online_retail_II.xlsx') else 'data/raw/online_retail_II.xlsx'
print(f"Data source path: {file_path}")

# Inspect available sheets
excel_book = pd.ExcelFile(file_path)
print("Available Excel sheets:", excel_book.sheet_names)

# Load Sheet 1
print("\nLoading Sheet: Year 2009-2010...")
df_2009_2010 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
print(f"Year 2009-2010 shape: {df_2009_2010.shape[0]:,} rows x {df_2009_2010.shape[1]} columns")

# Load Sheet 2
print("Loading Sheet: Year 2010-2011...")
df_2010_2011 = pd.read_excel(file_path, sheet_name='Year 2010-2011')
print(f"Year 2010-2011 shape: {df_2010_2011.shape[0]:,} rows x {df_2010_2011.shape[1]} columns")

# Combine both sheets into df_raw
df_raw = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
print(f"\nConsolidated df_raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Data source path: ../data/raw/online_retail_II.xlsx


Available Excel sheets: ['Year 2009-2010', 'Year 2010-2011']

Loading Sheet: Year 2009-2010...


Year 2009-2010 shape: 525,461 rows x 8 columns
Loading Sheet: Year 2010-2011...


Year 2010-2011 shape: 541,910 rows x 8 columns

Consolidated df_raw shape: 1,067,371 rows x 8 columns


## A. Dataset Overview

In this section, we examine the structural characteristics of `df_raw`:
- Total rows and columns.
- Data types and non-null counts.
- Exact memory consumption.
- First and last five records.
- Chronological date range.
- Cardinality of products, invoices, and countries.

In [3]:
print("=" * 60)
print("A.1 DATASET DIMENSIONS & SCHEMA")
print("=" * 60)
print(f"Total Rows:    {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]:,}")

print("\n--- DataFrame Information ---")
df_raw.info(memory_usage='deep')

# Explicit deep memory calculation
deep_memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"\nTotal Deep Memory Footprint: {deep_memory_mb:.2f} MB")

A.1 DATASET DIMENSIONS & SCHEMA
Total Rows:    1,067,371
Total Columns: 8

--- DataFrame Information ---


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 249.1 MB



Total Deep Memory Footprint: 249.05 MB


In [4]:
print("=" * 60)
print("A.2 FIRST AND LAST FIVE RECORDS")
print("=" * 60)

print("\nFirst 5 Records (Head):")
display(df_raw.head())

print("\nLast 5 Records (Tail):")
display(df_raw.tail())

A.2 FIRST AND LAST FIVE RECORDS

First 5 Records (Head):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.9500,13085.0000,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085.0000,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.7500,13085.0000,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1000,13085.0000,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.2500,13085.0000,United Kingdom



Last 5 Records (Tail):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.1000,12680.0000,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.1500,12680.0000,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.1500,12680.0000,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.9500,12680.0000,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.0000,12680.0000,France


In [5]:
print("=" * 60)
print("A.3 DATE RANGE & CORE CARDINALITY")
print("=" * 60)

min_date = df_raw['InvoiceDate'].min()
max_date = df_raw['InvoiceDate'].max()
timespan = max_date - min_date

print(f"Earliest Invoice Date: {min_date}")
print(f"Latest Invoice Date:   {max_date}")
print(f"Total Date Range:      {timespan.days} days ({timespan.days / 365.25:.2f} years)")

# Cardinality
unique_invoices = df_raw['Invoice'].nunique()
unique_products_stock = df_raw['StockCode'].nunique()
unique_products_desc = df_raw['Description'].nunique()
unique_countries = df_raw['Country'].nunique()

print(f"\nUnique Invoices:         {unique_invoices:,}")
print(f"Unique Stock Codes:      {unique_products_stock:,}")
print(f"Unique Descriptions:     {unique_products_desc:,}")
print(f"Unique Countries:        {unique_countries:,}")

A.3 DATE RANGE & CORE CARDINALITY
Earliest Invoice Date: 2009-12-01 07:45:00
Latest Invoice Date:   2011-12-09 12:50:00
Total Date Range:      738 days (2.02 years)



Unique Invoices:         53,628
Unique Stock Codes:      5,305
Unique Descriptions:     5,698
Unique Countries:        43


## B. Missing Values Analysis

Here we evaluate missing values across all columns:
- Missing value counts and percentages per column.
- Total unique rows containing one or more missing fields.
- Overlap analysis between missing `Customer ID` and missing `Description`.

In [6]:
print("=" * 60)
print("B. MISSING VALUES PER COLUMN")
print("=" * 60)

missing_count = df_raw.isnull().sum()
missing_pct = (missing_count / len(df_raw)) * 100

missing_summary = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing Count': missing_count.values,
    'Missing Percentage (%)': missing_pct.values,
    'Data Type': df_raw.dtypes.values
}).sort_values(by='Missing Count', ascending=False)

display(missing_summary.reset_index(drop=True))

# Total rows with any missing value
rows_with_missing = df_raw.isnull().any(axis=1).sum()
pct_rows_missing = (rows_with_missing / len(df_raw)) * 100
print(f"\nTotal rows containing at least one missing value: {rows_with_missing:,} ({pct_rows_missing:.2f}%)")

# Overlap between missing Description and missing Customer ID
both_missing = (df_raw['Description'].isnull() & df_raw['Customer ID'].isnull()).sum()
desc_missing_cust_present = (df_raw['Description'].isnull() & df_raw['Customer ID'].notnull()).sum()
print(f"Rows missing both Description and Customer ID: {both_missing:,} ({both_missing / missing_count['Description'] * 100:.2f}% of missing descriptions)")
print(f"Rows missing Description but having Customer ID: {desc_missing_cust_present:,}")

B. MISSING VALUES PER COLUMN


,Column,Missing Count,Missing Percentage (%),Data Type
0,Customer ID,243007,22.7669,float64
1,Description,4382,0.4105,object
2,StockCode,0,0.0000,object
3,Invoice,0,0.0000,object
4,Quantity,0,0.0000,int64
5,InvoiceDate,0,0.0000,datetime64[us]
6,Price,0,0.0000,float64
7,Country,0,0.0000,str



Total rows containing at least one missing value: 243,007 (22.77%)
Rows missing both Description and Customer ID: 4,382 (100.00% of missing descriptions)
Rows missing Description but having Customer ID: 0


## C. Duplicate Analysis

Exact duplicate rows (where all 8 columns share identical values) often occur due to:
- Multi-click checkouts or network retry requests.
- Repeated manual batch scanner entries.
- Splitting line items across identical orders.

We quantify the exact duplicate rows across all columns.

In [7]:
print("=" * 60)
print("C. EXACT DUPLICATE ROWS ANALYSIS")
print("=" * 60)

total_duplicates = df_raw.duplicated().sum()
duplicate_pct = (total_duplicates / len(df_raw)) * 100

print(f"Total Exact Duplicate Rows: {total_duplicates:,}")
print(f"Duplicate Percentage:       {duplicate_pct:.2f}%")

# Sample duplicate rows sorted together to show pairs
duplicate_sample = df_raw[df_raw.duplicated(keep=False)].sort_values(by=['Invoice', 'StockCode']).head(10)
print("\nSample of Duplicate Line Items:")
display(duplicate_sample)

C. EXACT DUPLICATE ROWS ANALYSIS


Total Exact Duplicate Rows: 34,335
Duplicate Percentage:       3.22%



Sample of Duplicate Line Items:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.9500,16329.0000,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.9500,16329.0000,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.7500,16329.0000,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.8500,16329.0000,United Kingdom


## D. Transaction Analysis

Retail transaction logs contain operational edge cases that must be distinguished before modeling:
1. **Cancelled Invoices:** Invoices prefixed with `'C'` indicate transaction cancellations or credit notes.
2. **Negative Quantities:** May represent customer returns or internal warehouse stock write-downs.
3. **Zero Quantities:** Test for potential voided or placeholder rows.
4. **Negative Prices:** Examine whether negative prices reflect customer discounts or accounting ledger adjustments.
5. **Zero Prices:** Determine whether zero-priced rows represent promotional free gifts or internal inventory adjustments.

In [8]:
print("=" * 60)
print("D.1 CANCELLED INVOICES ANALYSIS")
print("=" * 60)

# Convert Invoice to string to inspect prefixes
invoice_str = df_raw['Invoice'].astype(str)
cancelled_mask = invoice_str.str.startswith('C')

cancelled_rows = cancelled_mask.sum()
cancelled_pct = (cancelled_rows / len(df_raw)) * 100
unique_cancelled_invoices = invoice_str[cancelled_mask].nunique()

print(f"Cancelled Invoice Rows (prefix 'C'): {cancelled_rows:,} ({cancelled_pct:.2f}%)")
print(f"Unique Cancelled Invoice IDs:        {unique_cancelled_invoices:,}")

print("\nSample Cancelled Invoices:")
display(df_raw[cancelled_mask].head(5))

D.1 CANCELLED INVOICES ANALYSIS


Cancelled Invoice Rows (prefix 'C'): 19,494 (1.83%)
Unique Cancelled Invoice IDs:        8,292

Sample Cancelled Invoices:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.9500,16321.0000,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.6500,16321.0000,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.2500,16321.0000,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.1000,16321.0000,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.9500,16321.0000,Australia


In [9]:
print("=" * 60)
print("D.2 QUANTITY ANOMALIES (NEGATIVE & ZERO)")
print("=" * 60)

neg_qty_mask = df_raw['Quantity'] < 0
zero_qty_mask = df_raw['Quantity'] == 0

total_neg_qty = neg_qty_mask.sum()
neg_qty_pct = (total_neg_qty / len(df_raw)) * 100

total_zero_qty = zero_qty_mask.sum()
zero_qty_pct = (total_zero_qty / len(df_raw)) * 100

print(f"Total Negative Quantities (< 0): {total_neg_qty:,} ({neg_qty_pct:.2f}%)")
print(f"Total Zero Quantities (== 0):    {total_zero_qty:,} ({zero_qty_pct:.2f}%)")

# Detailed cross-tabulation between Cancelled Invoices and Quantity sign
canc_with_neg = (cancelled_mask & neg_qty_mask).sum()
canc_with_pos = (cancelled_mask & (df_raw['Quantity'] > 0)).sum()
canc_with_zero = (cancelled_mask & zero_qty_mask).sum()
non_canc_with_neg = ((~cancelled_mask) & neg_qty_mask).sum()

print("\n--- Cross-Tabulation: Cancellations vs Quantities ---")
print(f"Cancelled ('C') with Negative Quantity:     {canc_with_neg:,} (Legitimate customer returns/refunds)")
print(f"Cancelled ('C') with Positive Quantity:     {canc_with_pos:,} (Manual entry adjustment)")
print(f"Cancelled ('C') with Zero Quantity:         {canc_with_zero:,}")
print(f"Non-Cancelled with Negative Quantity:       {non_canc_with_neg:,} (Internal warehouse/inventory write-downs)")

# Inspecting non-cancelled negative quantity rows
non_canc_neg_records = df_raw[(~cancelled_mask) & neg_qty_mask]
print(f"\nCustomer ID presence in non-cancelled negative quantity rows: {non_canc_neg_records['Customer ID'].notnull().sum()} non-null")
print(f"Price == 0.0 in non-cancelled negative quantity rows:        {(non_canc_neg_records['Price'] == 0).sum():,} / {len(non_canc_neg_records):,}")
print("\nTop Descriptions for Non-Cancelled Negative Quantity Rows (Inventory Adjustments):")
display(non_canc_neg_records['Description'].value_counts().head(10))

D.2 QUANTITY ANOMALIES (NEGATIVE & ZERO)
Total Negative Quantities (< 0): 22,950 (2.15%)
Total Zero Quantities (== 0):    0 (0.00%)

--- Cross-Tabulation: Cancellations vs Quantities ---
Cancelled ('C') with Negative Quantity:     19,493 (Legitimate customer returns/refunds)
Cancelled ('C') with Positive Quantity:     1 (Manual entry adjustment)
Cancelled ('C') with Zero Quantity:         0
Non-Cancelled with Negative Quantity:       3,457 (Internal warehouse/inventory write-downs)

Customer ID presence in non-cancelled negative quantity rows: 0 non-null
Price == 0.0 in non-cancelled negative quantity rows:        3,457 / 3,457

Top Descriptions for Non-Cancelled Negative Quantity Rows (Inventory Adjustments):


Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64

In [10]:
print("=" * 60)
print("D.3 PRICE ANOMALIES (NEGATIVE & ZERO)")
print("=" * 60)

neg_price_mask = df_raw['Price'] < 0
zero_price_mask = df_raw['Price'] == 0

total_neg_price = neg_price_mask.sum()
neg_price_pct = (total_neg_price / len(df_raw)) * 100

total_zero_price = zero_price_mask.sum()
zero_price_pct = (total_zero_price / len(df_raw)) * 100

print(f"Total Negative Prices (< 0): {total_neg_price:,} ({neg_price_pct:.4f}%)")
print(f"Total Zero Prices (== 0):    {total_zero_price:,} ({zero_price_pct:.2f}%)")

# Inspect all negative price rows
print("\n--- All Records with Negative Unit Price ---")
display(df_raw[neg_price_mask])

# Inspect zero price rows breakdown
zero_price_records = df_raw[zero_price_mask]
zero_price_no_cust = zero_price_records['Customer ID'].isnull().sum()
zero_price_with_cust = zero_price_records['Customer ID'].notnull().sum()

print("\n--- Zero Price Classification ---")
print(f"Zero Price without Customer ID: {zero_price_no_cust:,} (Internal logs: damages, checks, lost, stock corrections)")
print(f"Zero Price with Customer ID:    {zero_price_with_cust:,} (Promotional items / customer free samples)")
print("\nTop Descriptions for Zero Price Records:")
display(zero_price_records['Description'].value_counts().head(10))

D.3 PRICE ANOMALIES (NEGATIVE & ZERO)
Total Negative Prices (< 0): 5 (0.0005%)
Total Zero Prices (== 0):    6,202 (0.58%)

--- All Records with Negative Unit Price ---


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.3600,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.7900,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.8700,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.0600,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.0600,NaN,United Kingdom



--- Zero Price Classification ---
Zero Price without Customer ID: 6,131 (Internal logs: damages, checks, lost, stock corrections)
Zero Price with Customer ID:    71 (Promotional items / customer free samples)

Top Descriptions for Zero Price Records:


Description
check                    162
?                         92
damages                   84
damaged                   81
found                     28
missing                   27
sold as set on dotcom     20
Damaged                   17
adjustment                16
OWL DOORSTOP              15
Name: count, dtype: int64

## E. Business Overview

In this section, we analyze the core commercial dimensions of the dataset:
- Total unique invoices.
- Total unique registered customers (excluding null customer identifiers).
- Total unique product catalog items.
- Geographic distribution of transactions.
- Statistical distribution of transaction quantities and unit prices.

In [11]:
print("=" * 60)
print("E.1 CORE BUSINESS METRICS & GEOGRAPHY")
print("=" * 60)

total_unique_invoices = df_raw['Invoice'].nunique()
total_unique_customers = df_raw['Customer ID'].dropna().nunique()
total_unique_products = df_raw['StockCode'].nunique()

print(f"Total Unique Invoices:             {total_unique_invoices:,}")
print(f"Total Unique Registered Customers: {total_unique_customers:,}")
print(f"Total Unique Product SKUs:         {total_unique_products:,}")

# Country distribution table
country_counts = df_raw['Country'].value_counts()
country_pcts = (country_counts / len(df_raw)) * 100

country_summary_df = pd.DataFrame({
    'Transactions': country_counts,
    'Percentage (%)': country_pcts
})

print(f"\nGeographic Footprint: Top 15 Countries (out of {len(country_counts)} total):")
display(country_summary_df.head(15))

E.1 CORE BUSINESS METRICS & GEOGRAPHY
Total Unique Invoices:             53,628
Total Unique Registered Customers: 5,942
Total Unique Product SKUs:         5,305



Geographic Footprint: Top 15 Countries (out of 43 total):


,Transactions,Percentage (%)
Country,,
United Kingdom,981330,91.9390
EIRE,17866,1.6738
Germany,17624,1.6512
France,14330,1.3426
Netherlands,5140,0.4816
Spain,3811,0.3570
Switzerland,3189,0.2988
Belgium,3123,0.2926
Portugal,2620,0.2455


In [12]:
print("=" * 60)
print("E.2 QUANTITY & UNIT PRICE DISTRIBUTIONS")
print("=" * 60)

percentiles = [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]

print("--- Quantity Distribution ---")
qty_dist = df_raw['Quantity'].describe(percentiles=percentiles)
display(qty_dist.to_frame(name='Quantity'))

print("\n--- Unit Price Distribution ---")
price_dist = df_raw['Price'].describe(percentiles=percentiles)
display(price_dist.to_frame(name='Price'))

E.2 QUANTITY & UNIT PRICE DISTRIBUTIONS
--- Quantity Distribution ---


,Quantity
count,1067371.0000
mean,9.9389
std,172.7058
min,-80995.0000
0.1%,-112.6300
1%,-3.0000
5%,1.0000
25%,1.0000
50%,3.0000
75%,10.0000



--- Unit Price Distribution ---


,Price
count,1067371.0000
mean,4.6494
std,123.5531
min,-53594.3600
0.1%,0.0000
1%,0.2100
5%,0.4200
25%,1.2500
50%,2.1000
75%,4.1500


## F. Data-Quality Summary Table

Below is the consolidated data-quality audit summary table detailing every observed issue, its exact count, percentage relative to the consolidated dataset ($1,067,371$ rows), and actionable observations.

In [13]:
# Construct final audit summary table
audit_data = [
    {
        'Issue': 'Missing Customer IDs',
        'Count': int(df_raw['Customer ID'].isnull().sum()),
        'Percentage': (df_raw['Customer ID'].isnull().sum() / len(df_raw)) * 100,
        'Observation': 'Anonymous or guest checkouts; cannot be imputed without secondary linkage keys.'
    },
    {
        'Issue': 'Missing Descriptions',
        'Count': int(df_raw['Description'].isnull().sum()),
        'Percentage': (df_raw['Description'].isnull().sum() / len(df_raw)) * 100,
        'Observation': '100% overlap with missing Customer IDs; administrative placeholders with zero price.'
    },
    {
        'Issue': 'Duplicate Rows',
        'Count': int(df_raw.duplicated().sum()),
        'Percentage': (df_raw.duplicated().sum() / len(df_raw)) * 100,
        'Observation': 'Exact duplicates across all 8 columns from repeated logging or split line items.'
    },
    {
        'Issue': 'Cancelled Invoices',
        'Count': int(df_raw['Invoice'].astype(str).str.startswith('C').sum()),
        'Percentage': (df_raw['Invoice'].astype(str).str.startswith('C').sum() / len(df_raw)) * 100,
        'Observation': 'Invoices prefixed with C representing credit notes and customer returns (8,292 unique invoices).'
    },
    {
        'Issue': 'Negative Quantities',
        'Count': int((df_raw['Quantity'] < 0).sum()),
        'Percentage': ((df_raw['Quantity'] < 0).sum() / len(df_raw)) * 100,
        'Observation': '19,493 are customer returns (prefix C); 3,457 are inventory write-downs/adjustments.'
    },
    {
        'Issue': 'Zero Quantities',
        'Count': int((df_raw['Quantity'] == 0).sum()),
        'Percentage': ((df_raw['Quantity'] == 0).sum() / len(df_raw)) * 100,
        'Observation': 'No zero quantity transactions exist in the raw dataset.'
    },
    {
        'Issue': 'Negative Prices',
        'Count': int((df_raw['Price'] < 0).sum()),
        'Percentage': ((df_raw['Price'] < 0).sum() / len(df_raw)) * 100,
        'Observation': 'Invoices prefixed with A (Adjust bad debt) representing accounting write-offs, not retail sales.'
    },
    {
        'Issue': 'Zero Prices',
        'Count': int((df_raw['Price'] == 0).sum()),
        'Percentage': ((df_raw['Price'] == 0).sum() / len(df_raw)) * 100,
        'Observation': '6,131 internal stock adjustments (damages, checks, samples) + 71 customer promotional gifts.'
    }
]

df_audit_summary = pd.DataFrame(audit_data)

# Format for clear tabular presentation
df_audit_display = df_audit_summary.copy()
df_audit_display['Count'] = df_audit_display['Count'].apply(lambda x: f"{x:,}")
df_audit_display['Percentage'] = df_audit_display['Percentage'].apply(lambda x: f"{x:.4f}%" if 0 < x < 0.01 else f"{x:.2f}%")

print("=" * 80)
print("FINAL DATA QUALITY SUMMARY TABLE")
print("=" * 80)
display(df_audit_display)

FINAL DATA QUALITY SUMMARY TABLE


,Issue,Count,Percentage,Observation
0,Missing Customer IDs,"243,007",22.77%,Anonymous or guest checkouts; cannot be impute...
1,Missing Descriptions,"4,382",0.41%,100% overlap with missing Customer IDs; admini...
2,Duplicate Rows,"34,335",3.22%,Exact duplicates across all 8 columns from rep...
3,Cancelled Invoices,"19,494",1.83%,Invoices prefixed with C representing credit n...
4,Negative Quantities,"22,950",2.15%,"19,493 are customer returns (prefix C); 3,457 ..."
5,Zero Quantities,0,0.00%,No zero quantity transactions exist in the raw...
6,Negative Prices,5,0.0005%,Invoices prefixed with A (Adjust bad debt) rep...
7,Zero Prices,"6,202",0.58%,"6,131 internal stock adjustments (damages, che..."


### Final Data-Quality Summary Table (Markdown)

| Issue | Count | Percentage | Observation |
|---|---:|---:|---|
| Missing Customer IDs | 243,007 | 22.77% | Anonymous or guest checkouts; cannot be imputed without secondary linkage keys. |
| Missing Descriptions | 4,382 | 0.41% | 100% overlap with missing Customer IDs; administrative placeholders with zero price. |
| Duplicate Rows | 34,335 | 3.22% | Exact duplicates across all 8 columns from repeated logging or split line items. |
| Cancelled Invoices | 19,494 | 1.83% | Invoices prefixed with 'C' representing credit notes and customer returns (8,292 unique invoices). |
| Negative Quantities | 22,950 | 2.15% | 19,493 are customer returns (prefix 'C'); 3,457 are inventory write-downs/adjustments. |
| Zero Quantities | 0 | 0.00% | No zero quantity transactions exist in the raw dataset. |
| Negative Prices | 5 | 0.0005% | Invoices prefixed with 'A' ('Adjust bad debt') representing accounting write-offs, not retail sales. |
| Zero Prices | 6,202 | 0.58% | 6,131 internal stock adjustments (damages, checks, samples) + 71 customer promotional gifts. |

---

### Audit Takeaways & Strategy for Subsequent Cleaning Phase
1. **Raw Dataset Integrity:** The combined dataset holds exactly **1,067,371 rows and 8 columns** spanning 2009-12-01 to 2011-12-09. All records remain untouched.
2. **Returns vs. Inventory Adjustments:** Negative quantities should **not** be globally removed. The 19,493 rows with invoice prefix `'C'` reflect customer returns which are critical for net revenue and return-rate calculations. Conversely, the 3,457 non-cancelled negative quantity rows represent inventory corrections (damages, lost stock, write-offs) with missing customer IDs and zero prices.
3. **Missing Customer IDs:** Approximately 22.77% of rows lack a `Customer ID`. While these transactions cannot be utilized for customer-level RFM or cohort analysis, they represent valid macro-level retail sales that must be factored into total sales volume unless specifically doing customer-level analytics.
4. **Accounting Adjustments:** The 5 negative price rows are confirmed debt adjustments (`StockCode: 'B'`, `Description: 'Adjust bad debt'`) and should be quarantined from product sales metrics.
5. **Duplicate Records:** 34,335 rows are identical across every column and can be safely deduplicated in the subsequent cleaning stage.